# Note: Some engineered features are unusable because they directly interact with sale price. Using the model on these features would be disingenuous, so the following columns must be dropped when training and evaluating models:
1) Assessment ratio
2) Assessment gap
3) Town Avg Sale Amount
4) Town Median Sale Amount
5) Luxury Home

In [1]:
# imports
import pandas as pd
import numpy as np
import re
from math import radians, sin, cos, sqrt, atan2

In [2]:
df = pd.read_csv("../data/real-estate-cleaned.csv")
df.head()

,List Year,Date Recorded,Town,Assessed Value,Sale Amount,Property Type,Residential Type,Location
0,2010,10/2/10,Norwalk,339640.0,265000.0,Residential,Single Family,POINT (-73.408 41.118)
1,2010,10/3/10,Milford,674350.0,788000.0,Residential,Single Family,POINT (-73.057 41.222)
2,2010,10/4/10,Bridgeport,132250.0,148000.0,Residential,Single Family,POINT (-73.195 41.186)
3,2010,10/4/10,Bristol,99610.0,32000.0,Residential,Single Family,POINT (-72.94 41.671)
4,2010,10/4/10,Bridgeport,132260.0,110000.0,Residential,Single Family,POINT (-73.195 41.186)


In [3]:
print(df.columns)
print(df.shape)

Index(['List Year', 'Date Recorded', 'Town', 'Assessed Value', 'Sale Amount',
       'Property Type', 'Residential Type', 'Location'],
      dtype='str')
(614534, 8)


In [4]:
'''
convert plaintext date representations into real datetime values
'''

df["Date Recorded"] = pd.to_datetime(df["Date Recorded"], errors="coerce")
df[["Date Recorded"]].head()

/var/folders/nm/q5_ndlm10hz8fcjsbx6q_g200000gn/T/ipykernel_88773/402475579.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date Recorded"] = pd.to_datetime(df["Date Recorded"], errors="coerce")


,Date Recorded
0,2010-10-02
1,2010-10-03
2,2010-10-04
3,2010-10-04
4,2010-10-04


In [5]:
'''
note several features based on sale month. primary focus is to provide additional
context for seasonal impactors.
'''

df["Sale Year"] = df["Date Recorded"].dt.year
df["Sale Month"] = df["Date Recorded"].dt.month
df["Sale Quarter"] = df["Date Recorded"].dt.quarter
df["Sale Day Of Year"] = df["Date Recorded"].dt.dayofyear

df[["Date Recorded", "Sale Year", "Sale Month", "Sale Quarter", "Sale Day Of Year"]].head()

,Date Recorded,Sale Year,Sale Month,Sale Quarter,Sale Day Of Year
0,2010-10-02,2010,10,4,275
1,2010-10-03,2010,10,4,276
2,2010-10-04,2010,10,4,277
3,2010-10-04,2010,10,4,277
4,2010-10-04,2010,10,4,277


In [6]:
'''
calculate assessment ratio and gap in order to provide context for how much
properties were sold for relative to their assessed value. log assessed value
reduces skew in large value ranges.
'''

df["Assessment Ratio"] = np.where(
    df["Assessed Value"] > 0,
    df["Sale Amount"] / df["Assessed Value"],
    np.nan
)

# houses without an assessment ratio are stored as 0 in the csv.
# as such, they will not have an assessment ratio calculated.

df["Assessment Gap"] = df["Sale Amount"] - df["Assessed Value"]
df["Log Assessed Value"] = np.log1p(df["Assessed Value"])

df[["Assessed Value", "Sale Amount", "Assessment Ratio", "Assessment Gap", "Log Assessed Value"]].head()

,Assessed Value,Sale Amount,Assessment Ratio,Assessment Gap,Log Assessed Value
0,339640.0,265000.0,0.780238,-74640.0,12.735644
1,674350.0,788000.0,1.168533,113650.0,13.421506
2,132250.0,148000.0,1.119093,15750.0,11.792457
3,99610.0,32000.0,0.321253,-67610.0,11.509028
4,132260.0,110000.0,0.831695,-22260.0,11.792533


In [7]:
'''
extract latitude and longitude into separate numeric columns
'''

def extract_coordinates(point_str):
    if pd.isna(point_str):
        return np.nan, np.nan

    match = re.match(r"POINT \(([-0-9.]+) ([-0-9.]+)\)", str(point_str))
    if match:
        lon = float(match.group(1))
        lat = float(match.group(2))
        return lat, lon

    return np.nan, np.nan

df[["Latitude", "Longitude"]] = df["Location"].apply(
    lambda x: pd.Series(extract_coordinates(x))
)

df[["Location", "Latitude", "Longitude"]].head()

,Location,Latitude,Longitude
0,POINT (-73.408 41.118),41.118,-73.408
1,POINT (-73.057 41.222),41.222,-73.057
2,POINT (-73.195 41.186),41.186,-73.195
3,POINT (-72.94 41.671),41.671,-72.940
4,POINT (-73.195 41.186),41.186,-73.195


In [8]:
'''
computes approximate distance in miles between two sets of coordinates
'''

def haversine(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 3958.8  # Earth radius in miles

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [9]:
'''
computes approximate distance to major cities in and around Connecticut.
New York City, Hartford, and Boston were chosen because they may have an
impact on housing prices. Boston and NYC are major cities, and Hartford
is a centrally-accessible location.
'''

NYC_LAT, NYC_LON = 40.7128, -74.0060
HARTFORD_LAT, HARTFORD_LON = 41.7658, -72.6734
BOSTON_LAT, BOSTON_LON = 42.3601, -71.0589

df["Dist To NYC"] = df.apply(
    lambda row: haversine(row["Latitude"], row["Longitude"], NYC_LAT, NYC_LON), axis=1
)

df["Dist To Hartford"] = df.apply(
    lambda row: haversine(row["Latitude"], row["Longitude"], HARTFORD_LAT, HARTFORD_LON), axis=1
)

df["Dist To Boston"] = df.apply(
    lambda row: haversine(row["Latitude"], row["Longitude"], BOSTON_LAT, BOSTON_LON), axis=1
)

df[["Latitude", "Longitude", "Dist To NYC", "Dist To Hartford", "Dist To Boston"]].head()

,Latitude,Longitude,Dist To NYC,Dist To Hartford,Dist To Boston
0,41.118,-73.408,41.936910,58.745280,148.427562
1,41.222,-73.057,60.737843,42.495549,129.526272
2,41.186,-73.195,53.480470,48.310972,136.729259
3,41.671,-72.940,86.341970,15.229990,107.661631
4,41.186,-73.195,53.480470,48.310972,136.729259


In [10]:
'''
since precise coastline geometry is not present, we use a general coastal
latitude in order to represent and calculate distance to the coast. houses
close to the coast likely have effects on their price as a result.
'''

CT_COAST_LAT = 41.0

df["Approx Dist To Coast"] = df["Latitude"].apply(
    lambda lat: np.nan if pd.isna(lat) else abs(lat - CT_COAST_LAT) * 69
)

df[["Latitude", "Approx Dist To Coast"]].head()

,Latitude,Approx Dist To Coast
0,41.118,8.142
1,41.222,15.318
2,41.186,12.834
3,41.671,46.299
4,41.186,12.834


**The following 3 cells are used to calculate town median sale amount, town average assessment ratio, and town average sale amount, respectively**

In [11]:
df["Town Median Sale Amount"] = df.groupby("Town")["Sale Amount"].transform("median")
df[["Town", "Sale Amount", "Town Median Sale Amount"]].head()

,Town,Sale Amount,Town Median Sale Amount
0,Norwalk,265000.0,420000.0
1,Milford,788000.0,287000.0
2,Bridgeport,148000.0,160000.0
3,Bristol,32000.0,185000.0
4,Bridgeport,110000.0,160000.0


In [12]:
df["Town Avg Assessment Ratio"] = df.groupby("Town")["Assessment Ratio"].transform("mean")
df[["Town", "Assessment Ratio", "Town Avg Assessment Ratio"]].head()

,Town,Assessment Ratio,Town Avg Assessment Ratio
0,Norwalk,0.780238,1.876569
1,Milford,1.168533,1.496553
2,Bridgeport,1.119093,1.492121
3,Bristol,0.321253,1.731108
4,Bridgeport,0.831695,1.492121


In [13]:
df["Town Avg Sale Amount"] = df.groupby("Town")["Sale Amount"].transform("mean")
df[["Town", "Sale Amount", "Town Avg Sale Amount"]].head()

,Town,Sale Amount,Town Avg Sale Amount
0,Norwalk,265000.0,617242.675586
1,Milford,788000.0,344579.129171
2,Bridgeport,148000.0,212777.367623
3,Bristol,32000.0,211111.211163
4,Bridgeport,110000.0,212777.367623


In [14]:
'''
computes relative market strength compared to median values based on town
'''

overall_median_sale = df["Sale Amount"].median()

df["Town Market Strength"] = df["Town Median Sale Amount"] / overall_median_sale

df[["Town", "Town Median Sale Amount", "Town Market Strength"]].head()

,Town,Town Median Sale Amount,Town Market Strength
0,Norwalk,420000.0,1.650295
1,Milford,287000.0,1.127701
2,Bridgeport,160000.0,0.628684
3,Bristol,185000.0,0.726916
4,Bridgeport,160000.0,0.628684


In [15]:
'''
create a threshold beyond which houses will be labelled as luxury homes. this should
help with outlier management.
'''

LUXURY_THRESHOLD = 2_000_000

df["Luxury Home"] = (df["Sale Amount"] >= LUXURY_THRESHOLD).astype(int)

df[["Sale Amount", "Luxury Home"]].head()
print(df["Luxury Home"].value_counts())

Luxury Home
0    602396
1     12138
Name: count, dtype: int64


# Sanity Checks

In [16]:
print(df.columns)

Index(['List Year', 'Date Recorded', 'Town', 'Assessed Value', 'Sale Amount',
       'Property Type', 'Residential Type', 'Location', 'Sale Year',
       'Sale Month', 'Sale Quarter', 'Sale Day Of Year', 'Assessment Ratio',
       'Assessment Gap', 'Log Assessed Value', 'Latitude', 'Longitude',
       'Dist To NYC', 'Dist To Hartford', 'Dist To Boston',
       'Approx Dist To Coast', 'Town Median Sale Amount',
       'Town Avg Assessment Ratio', 'Town Avg Sale Amount',
       'Town Market Strength', 'Luxury Home'],
      dtype='str')


In [17]:
engineered_cols = [
    "Sale Year",
    "Sale Month",
    "Sale Quarter",
    "Sale Day Of Year",
    "Assessment Ratio",
    "Assessment Gap",
    "Log Assessed Value",
    "Latitude",
    "Longitude",
    "Dist To NYC",
    "Dist To Hartford",
    "Dist To Boston",
    "Approx Dist To Coast",
    "Town Median Sale Amount",
    "Town Avg Assessment Ratio",
    "Town Avg Sale Amount",
    "Town Market Strength",
    "Luxury Home"
]

df[engineered_cols].head()

,Sale Year,Sale Month,Sale Quarter,Sale Day Of Year,Assessment Ratio,Assessment Gap,Log Assessed Value,Latitude,Longitude,Dist To NYC,Dist To Hartford,Dist To Boston,Approx Dist To Coast,Town Median Sale Amount,Town Avg Assessment Ratio,Town Avg Sale Amount,Town Market Strength,Luxury Home
0,2010,10,4,275,0.780238,-74640.0,12.735644,41.118,-73.408,41.936910,58.745280,148.427562,8.142,420000.0,1.876569,617242.675586,1.650295,0
1,2010,10,4,276,1.168533,113650.0,13.421506,41.222,-73.057,60.737843,42.495549,129.526272,15.318,287000.0,1.496553,344579.129171,1.127701,0
2,2010,10,4,277,1.119093,15750.0,11.792457,41.186,-73.195,53.480470,48.310972,136.729259,12.834,160000.0,1.492121,212777.367623,0.628684,0
3,2010,10,4,277,0.321253,-67610.0,11.509028,41.671,-72.940,86.341970,15.229990,107.661631,46.299,185000.0,1.731108,211111.211163,0.726916,0
4,2010,10,4,277,0.831695,-22260.0,11.792533,41.186,-73.195,53.480470,48.310972,136.729259,12.834,160000.0,1.492121,212777.367623,0.628684,0


In [18]:
df[engineered_cols].isna().sum()

# missing assessment ratios are okay because some properties did not have an assessed value.

Sale Year                       0
Sale Month                      0
Sale Quarter                    0
Sale Day Of Year                0
Assessment Ratio             1433
Assessment Gap                  0
Log Assessed Value              0
Latitude                        0
Longitude                       0
Dist To NYC                     0
Dist To Hartford                0
Dist To Boston                  0
Approx Dist To Coast            0
Town Median Sale Amount         0
Town Avg Assessment Ratio       0
Town Avg Sale Amount            0
Town Market Strength            0
Luxury Home                     0
dtype: int64

In [19]:
df.to_csv("../data/cleaned-engineered.csv", index=False)

# Final verification

In [20]:
check_df = pd.read_csv("../data/cleaned-engineered.csv")
check_df.head()

,List Year,Date Recorded,Town,Assessed Value,Sale Amount,Property Type,Residential Type,Location,Sale Year,Sale Month,...,Longitude,Dist To NYC,Dist To Hartford,Dist To Boston,Approx Dist To Coast,Town Median Sale Amount,Town Avg Assessment Ratio,Town Avg Sale Amount,Town Market Strength,Luxury Home
0,2010,2010-10-02,Norwalk,339640.0,265000.0,Residential,Single Family,POINT (-73.408 41.118),2010,10,...,-73.408,41.936910,58.745280,148.427562,8.142,420000.0,1.876569,617242.675586,1.650295,0
1,2010,2010-10-03,Milford,674350.0,788000.0,Residential,Single Family,POINT (-73.057 41.222),2010,10,...,-73.057,60.737843,42.495549,129.526272,15.318,287000.0,1.496553,344579.129171,1.127701,0
2,2010,2010-10-04,Bridgeport,132250.0,148000.0,Residential,Single Family,POINT (-73.195 41.186),2010,10,...,-73.195,53.480470,48.310972,136.729259,12.834,160000.0,1.492121,212777.367623,0.628684,0
3,2010,2010-10-04,Bristol,99610.0,32000.0,Residential,Single Family,POINT (-72.94 41.671),2010,10,...,-72.940,86.341970,15.229990,107.661631,46.299,185000.0,1.731108,211111.211163,0.726916,0
4,2010,2010-10-04,Bridgeport,132260.0,110000.0,Residential,Single Family,POINT (-73.195 41.186),2010,10,...,-73.195,53.480470,48.310972,136.729259,12.834,160000.0,1.492121,212777.367623,0.628684,0


In [ ]:
print(check_df.shape)
print(check_df.columns)

(614534, 26)
Index(['List Year', 'Date Recorded', 'Town', 'Assessed Value', 'Sale Amount',
       'Property Type', 'Residential Type', 'Location', 'Sale Year',
       'Sale Month', 'Sale Quarter', 'Sale Day Of Year', 'Assessment Ratio',
       'Assessment Gap', 'Log Assessed Value', 'Latitude', 'Longitude',
       'Dist To NYC', 'Dist To Hartford', 'Dist To Boston',
       'Approx Dist To Coast', 'Town Median Sale Amount',
       'Town Avg Assessment Ratio', 'Town Avg Sale Amount',
       'Town Market Strength', 'Luxury Home'],
      dtype='str')
